# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

# Voigt function

In [ ]:
from skewerjax.optical_depth.voigt import log10_voigt

### Numerical errors and comparison between implementations

In [ ]:
xx = jnp.linspace(-9, 9, 2**13)

sigma_low = 1e-3
sigma_high = 1.0
gamma_low = 1e-3
gamma_high = 1.0
sigma = jnp.array([sigma_low, sigma_high, sigma_low, sigma_high])
gamma = jnp.array([gamma_low, gamma_low, gamma_high, gamma_high])

Voigt_dict = {}

Voigt_dict['scipy_wofz'] = log10_voigt(xx, sigma, gamma, mode="scipy_wofz")

voigt_convolution_batched = vmap(lambda s, g: log10_voigt(xx, s, g, mode="convolution", oversample=4), in_axes=(0, 0))

Voigt_dict["JAX_convolution"] = voigt_convolution_batched(sigma, gamma)

voigt_exojax_batched = vmap(lambda s, g: log10_voigt(xx, s, g, mode="ExoJAX"), in_axes=(0, 0))

Voigt_dict["exojax"] = voigt_exojax_batched(sigma, gamma)

In [ ]:
P = len(sigma)
colors = np.array(plt.cm.get_cmap("tab10").colors[:P])

fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw=dict(height_ratios=[2, 1]))

for ii in range(P):
    ax.plot(xx, 10**Voigt_dict['scipy_wofz'][ii], color=colors[ii], linestyle="-", linewidth=1.0)
    ax.plot(xx, 10**Voigt_dict['JAX_convolution'][ii], color=colors[ii], linestyle="--", linewidth=2.)
    ax.plot(xx, 10**Voigt_dict['exojax'][ii], color=colors[ii], linestyle=":", linewidth=3.5)

for ii in range(P):
    ax_res.plot(
        xx, 10**Voigt_dict['JAX_convolution'][ii] / 10**Voigt_dict['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle="--", linewidth=1.0
    )
    ax_res.plot(
        xx, 10**Voigt_dict['exojax'][ii] / 10**Voigt_dict['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle=":", linewidth=2.0
    )

method_handles = [
    Line2D([0], [0], color="k", lw=1,  ls="-",  label="Scipy wofz"),
    Line2D([0], [0], color="k", lw=2,  ls="--", label="JAX convolution"),
    Line2D([0], [0], color="k", lw=3.5,  ls=":",  label="exojax"),
]
leg1 = ax.legend(handles=method_handles, loc="upper right", fontsize=16)
ax.add_artist(leg1)

param_handles = []
for ii in range(P):
    lbl = rf"$\sigma={sigma[ii]:g},\ \gamma={gamma[ii]:g}$"
    param_handles.append(Line2D([0], [0], color=colors[ii], lw=6, label=lbl))

leg2 = ax.legend(handles=param_handles, loc="upper left", fontsize=16)
ax.add_artist(leg2)

ax.set_title("Voigt profiles")
ax.set_ylabel(r"$V(x;\ \sigma,\ \gamma)$")
ax.set_yscale("log")
ax.set_ylim(1e-6, 1e3)

ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res.set_xlabel("x")        # only bottom panel gets the x-label
ax_res.set_ylabel("residual")
ax_res.set_yscale("symlog", linthresh=1e-6)

major_ticks = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res.set_yticks(major_ticks)

ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

plt.subplots_adjust(hspace=0.05)
fig.tight_layout()
plt.show()

### Autodiff

In [ ]:
xx = jnp.linspace(-9.0, 9.0, 2**12)  # frequency (or velocity) offset grid

sigma0 = 1.0
gamma0 = 0.001

In [ ]:
def voigt_scalar(x, sigma, gamma):
    return 10**log10_voigt(jnp.atleast_1d(x), sigma, gamma, mode="ExoJAX")[0]

# Derivative wrt x (ν)
dVoigt_dx = vmap(grad(voigt_scalar, argnums=0), in_axes=(0, None, None))

# Derivative wrt sigma (Gaussian width)
dVoigt_dsigma = vmap(grad(voigt_scalar, argnums=1), in_axes=(0, None, None))

# Derivative wrt sigma (Lorentzian width)
dVoigt_dgamma = vmap(grad(voigt_scalar, argnums=2), in_axes=(0, None, None))

dV_dx = dVoigt_dx(xx, sigma0, gamma0)
dV_dsigma = dVoigt_dsigma(xx, sigma0, gamma0)
dV_dgamma = dVoigt_dgamma(xx, sigma0, gamma0)

In [ ]:
delta_sigma = 0.2
sigma = jnp.linspace(sigma0 - delta_sigma, sigma0 + delta_sigma, 5)
sigma_ref_idx = len(sigma) // 2
sigma_ref = sigma[sigma_ref_idx]

V_sigma = vmap(lambda s: 10**log10_voigt(xx, s, gamma0), in_axes=(0,))(sigma)
V_sigma_ref = 10**log10_voigt(xx, sigma_ref, gamma0)

delta_gamma = 0.0009
gamma = jnp.linspace(gamma0 - delta_gamma, gamma0 + delta_gamma, 5)
gamma_ref_idx = len(gamma) // 2
gamma_ref = gamma[gamma_ref_idx]

V_gamma = vmap(lambda g: 10**log10_voigt(xx, sigma0, g), in_axes=(0,))(gamma)
V_gamma_ref = 10**log10_voigt(xx, sigma0, gamma_ref)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex='col', gridspec_kw=dict(height_ratios=[2, 1]))

ax_main_s, ax_main_g = axes[0, 0], axes[0, 1]
ax_res_s, ax_res_g = axes[1, 0], axes[1, 1]

# =============================================================================
# Sigma variation (left column)
# =============================================================================
colors_sigma = np.array(plt.colormaps["Reds"](np.linspace(0, 1, len(sigma)+2)))[1:-1]

for i in range(len(sigma)):
    ax_main_s.plot(xx, V_sigma[i], color=colors_sigma[i], linewidth=2.0, linestyle='-')

for i in range(len(sigma)):
    residual = V_sigma[i] / V_sigma_ref - 1
    ax_res_s.plot(xx, residual, color=colors_sigma[i], linewidth=2.0, linestyle='-')

# Legend
param_handles_s = []
for i, s_val in enumerate(sigma):
    param_handles_s.append(Line2D([0], [0], color=colors_sigma[i], lw=6, label=rf"$\sigma={s_val:.3g}$"))
leg2_s = ax_main_s.legend(handles=param_handles_s, loc="upper left", fontsize=16)
ax_main_s.add_artist(leg2_s)

# Styling
ax_main_s.set_title(r"Voigt profile variation with $\sigma$", pad=10)
ax_main_s.set_ylabel(r"$V(x;\ \sigma,\ \gamma_0)$")
ax_main_s.set_yscale("log")
V_min_s, V_max_s = V_sigma.min(), V_sigma.max()
ax_main_s.set_ylim(V_min_s * 0.5, V_max_s * 2.0)

ax_res_s.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res_s.set_xlabel("x")
ax_res_s.set_ylabel("residual")
ax_res_s.set_yscale("symlog", linthresh=1e-6)
major_ticks_s = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res_s.set_yticks(major_ticks_s)
ax_res_s.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res_s.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

# =============================================================================
# Gamma variation (right column)
# =============================================================================
colors_gamma = np.array(plt.colormaps["Greens"](np.linspace(0, 1, len(gamma)+2)))[1:-1]

for i in range(len(gamma)):
    ax_main_g.plot(xx, V_gamma[i], color=colors_gamma[i], linewidth=2.0, linestyle='-')

for i in range(len(gamma)):
    residual = V_gamma[i] / V_gamma_ref - 1
    ax_res_g.plot(xx, residual, color=colors_gamma[i], linewidth=2.0, linestyle='-')

# Legend
param_handles_g = []
for i, g_val in enumerate(gamma):
    param_handles_g.append(Line2D([0], [0], color=colors_gamma[i], lw=6, label=rf"$\gamma={g_val:.4g}$"))
leg2_g = ax_main_g.legend(handles=param_handles_g, loc="upper left", fontsize=16)
ax_main_g.add_artist(leg2_g)

# Styling
ax_main_g.set_title(r"Voigt profile variation with $\gamma$", pad=10)
ax_main_g.set_ylabel(r"$V(x;\ \sigma_0,\ \gamma)$")
ax_main_g.set_yscale("log")
V_min_g, V_max_g = V_gamma.min(), V_gamma.max()
ax_main_g.set_ylim(V_min_g * 0.5, V_max_g * 2.0)

ax_res_g.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res_g.set_xlabel("x")
ax_res_g.set_ylabel("residual")
ax_res_g.set_yscale("symlog", linthresh=1e-6)
major_ticks_g = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res_g.set_yticks(major_ticks_g)
ax_res_g.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res_g.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

# =============================================================================
fig.tight_layout()
fig.subplots_adjust(hspace=0.05, wspace=0.3)  # Add wspace for column separation

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

ax = axes[0]
ax.plot(xx, dV_dx)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dx")
ax.set_title("Derivative wrt x")

ax = axes[1]
ax.plot(xx, dV_dsigma)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dσ")
ax.set_title("Derivative wrt σ")

ax = axes[2]
ax.plot(xx, dV_dgamma)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dγ")
ax.set_title("Derivative wrt γ")

fig.tight_layout()
plt.show()